# Aether Stage 2 — Phase 2A frozen-Qwen R1 probe

This is the bounded ratio-1 probe from `docs/stage2_spec.md`: frozen Stage 1 AetherSpeech, frozen Qwen3-4B, and a trainable Connector at the native 12.5 Hz speech-state rate. It uses a fixed 4,096-example training subset and 256-example held-out subset. The hard budget is 5,000 optimizer steps.

Every task is isolated in its own cell. All artifacts are written immediately to a unique `MyDrive/aether-v3/stage2/runYYMMDD-HHMMSS/` directory.


## 0. Mount Google Drive

In [ ]:
import logging, os, subprocess, sys, time, json, shutil
from pathlib import Path
from google.colab import drive

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", force=True)
drive.mount("/content/drive")
print("DRIVE: mounted", flush=True)

## 1. Checkout the `stage2` branch and install

In [ ]:
REPO_URL = "https://github.com/karl4th/aether-v3.git"
REPO_DIR = "/content/aether-v3"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", "stage2"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", "stage2", "--single-branch", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "stage2"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/stage2"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR, "huggingface_hub"], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(Path(REPO_DIR)/"src"))
for module_name in list(sys.modules):
    if module_name == "aether_v3" or module_name.startswith("aether_v3."):
        del sys.modules[module_name]
GIT_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("CODE:", GIT_COMMIT, flush=True)

## 2. Read Hugging Face token

In [ ]:
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
assert os.environ["HF_TOKEN"], "Add HF_TOKEN in Colab Secrets and enable notebook access"
print("HF TOKEN: available (value hidden)", flush=True)

## 3. Load config and create the Drive run directory

In [ ]:
import torch
from aether_v3.config import load_config
from aether_v3.training.stage2_utils import create_run_dir

cfg = load_config("configs/stage2_phase2a.yaml")
assert cfg.llm.model_id == "Qwen/Qwen3-4B"
assert cfg.connector.resampler.ratio == 1
assert cfg.stage2_train.max_steps == 5000
assert cfg.stage2_train.batch_size == 1
assert torch.cuda.is_available(), "Select a GPU runtime"
if cfg.llm.dtype == "bfloat16" and not torch.cuda.is_bf16_supported():
    cfg.llm.dtype = "float16"
    cfg.stage2_train.amp_dtype = "float16"
run_dir = create_run_dir(cfg.stage2_train.drive_root)
cache_root = run_dir / "cache"
phase2a = {}
print("GPU:", torch.cuda.get_device_name(0), flush=True)
print("RUN:", run_dir, flush=True)
print("CONFIG: Qwen3-4B | ratio=1 | 4096 samples | 5000 steps", flush=True)

## 4. Download and load the private Stage 1 checkpoint

In [ ]:
from huggingface_hub import hf_hub_download
from aether_v3.models.aether_speech import AetherSpeechEncoder
from aether_v3.training.stage2_utils import load_stage1_encoder

stage1_path = hf_hub_download(
    repo_id=cfg.stage2_train.stage1_repo_id,
    filename=cfg.stage2_train.stage1_filename,
    revision=cfg.stage2_train.stage1_revision,
    token=os.environ["HF_TOKEN"],
)
encoder = AetherSpeechEncoder(cfg.aether_speech)
stage1_checkpoint = load_stage1_encoder(stage1_path, encoder)
encoder.eval()
print("STAGE 1:", stage1_path, "step=", stage1_checkpoint.get("step", "unknown"), flush=True)

## 5. Load tokenizer and Qwen3-4B model

In [ ]:
from transformers import AutoTokenizer
from aether_v3.models.aether_speech_llm import AetherSpeechLLM

tokenizer = AutoTokenizer.from_pretrained(cfg.llm.model_id, revision=cfg.llm.revision)
print("Loading Qwen3-4B...", flush=True)
model = AetherSpeechLLM(cfg.aether_speech, cfg.connector, cfg.llm, speech_frozen=True).cuda()
load_stage1_encoder(stage1_path, model.encoder)
model.connector.to(dtype=next(model.llm.parameters()).dtype)
assert model.connector.bridge.output_scale.dtype == torch.float32
print("MODEL: output_scale is FP32; exact initialization follows measured Qwen embedding RMS", flush=True)
print("MODEL: loaded; trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad), flush=True)

## 6. Build cached speech states: 4,096 train and 256 held-out

In [ ]:
from datasets import load_dataset
from aether_v3.data.stage2_cache import build_limmim_stage2_shards

train_rows = load_dataset("karl4th/limmim", split="train", streaming=True, token=os.environ["HF_TOKEN"])
val_rows = load_dataset("karl4th/limmim", split="validation", streaming=True, token=os.environ["HF_TOKEN"])
print("CACHE: building 4096 fixed train records", flush=True)
train_count = build_limmim_stage2_shards(train_rows, model.encoder, tokenizer, cache_root/"train", "train", max_examples=4096)
print("CACHE: building 256 fixed held-out records", flush=True)
val_count = build_limmim_stage2_shards(val_rows, model.encoder, tokenizer, cache_root/"validation", "validation", max_examples=256)
assert train_count == 4096 and val_count == 256
phase2a["cache_counts"] = {"train": train_count, "validation": val_count}
print("CACHE: PASS", phase2a["cache_counts"], flush=True)

## 7. Load one real B=2 diagnostic batch

In [ ]:
from torch.utils.data import DataLoader
from aether_v3.data.stage2_collate import collate_stage2_batch
from aether_v3.data.stage2_dataset import Stage2ShardDataset

diag_loader = DataLoader(Stage2ShardDataset(cache_root/"train", shuffle=False), batch_size=2, collate_fn=collate_stage2_batch)
batch_cpu = next(iter(diag_loader))
device = torch.device("cuda")
batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch_cpu.items()}
assert batch["speech_states"].shape[0] == 2
print("BATCH: PASS", {k: tuple(v.shape) for k,v in batch.items() if torch.is_tensor(v)}, flush=True)

## 8. Compare text and Connector embedding statistics before training

In [ ]:
def embedding_stats(x):
    x = x.detach().float().reshape(-1, x.shape[-1])
    l2 = x.norm(dim=-1)
    return {
        "mean": float(x.mean()), "std": float(x.std()),
        "rms": float(x.square().mean().sqrt()),
        "l2_p50": float(l2.quantile(0.50)), "l2_p90": float(l2.quantile(0.90)),
        "l2_p99": float(l2.quantile(0.99)),
    }

with torch.no_grad():
    text_ids = torch.cat([batch["prefix_ids"][batch["prefix_mask"]], batch["target_ids"][batch["target_mask"]]])
    text_embeds = model.llm.get_input_embeddings()(text_ids)
    text_stats = embedding_stats(text_embeds)
    initial_scale = text_stats["rms"]
    model.connector.bridge.output_scale.fill_(initial_scale)
    cfg.connector.bridge.init_output_scale = initial_scale
    connector_embeds, connector_mask = model.connector(batch["speech_states"].to(next(model.connector.parameters()).dtype), batch["speech_mask"])
    valid_connector = connector_embeds[connector_mask]
phase2a["embedding_stats"] = {"qwen_text": text_stats, "connector_pretrain": embedding_stats(valid_connector), "calibrated_output_scale": initial_scale}
ratio = phase2a["embedding_stats"]["connector_pretrain"]["rms"] / text_stats["rms"]
assert 0.9 <= ratio <= 1.1, f"Connector/Qwen RMS mismatch after calibration: {ratio:.3f}"
print(json.dumps(phase2a["embedding_stats"], indent=2), flush=True)
(run_dir/"embedding_stats.json").write_text(json.dumps(phase2a["embedding_stats"], indent=2))


## 9. One real forward: finite values, shapes, masks, and labels

In [ ]:
model.train()
speech_embeds, speech_mask = model.connector(batch["speech_states"].to(next(model.connector.parameters()).dtype), batch["speech_mask"])
built = model.build_inputs_embeds(batch["prefix_ids"], batch["prefix_mask"], speech_embeds, speech_mask, batch["target_ids"], batch["target_mask"])
out = model.forward_cached(batch)
assert out.loss is not None and torch.isfinite(out.loss)
assert torch.isfinite(out.logits).all()
assert out.logits.shape[:2] == built.labels.shape
assert built.attention_mask.dtype == torch.bool
assert torch.equal((built.labels != -100).sum(1), batch["target_mask"].sum(1))
for i in range(2):
    p, s, t = (int(batch["prefix_mask"][i].sum()), int(batch["speech_mask"][i].sum()), int(batch["target_mask"][i].sum()))
    total = p + 1 + s + 1 + t
    assert int(built.attention_mask[i].sum()) == total
    print(f"sample={i} prefix_len={p} speech_len={s} target_len={t} total_len={total} labels={int((built.labels[i] != -100).sum())}", flush=True)
phase2a["forward"] = {"loss": float(out.loss.detach()), "logits_shape": list(out.logits.shape), "passed": True}
print("FORWARD: PASS", phase2a["forward"], flush=True)

## 10. One backward: Connector and boundary gradients only

In [ ]:
model.zero_grad(set_to_none=True)
out = model.forward_cached(batch)
out.loss.backward()
connector_grad_norm = float(torch.sqrt(sum((p.grad.float().square().sum() for p in model.connector.parameters() if p.grad is not None))))
start_grad = float(model.connector.speech_start.grad.float().norm())
end_grad = float(model.connector.speech_end.grad.float().norm())
qwen_has_grad = any(p.grad is not None for p in model.llm.parameters())
encoder_has_grad = any(p.grad is not None for p in model.encoder.parameters())
assert connector_grad_norm > 0 and start_grad > 0 and end_grad > 0
assert not qwen_has_grad and not encoder_has_grad
phase2a["backward"] = {"connector_grad_norm": connector_grad_norm, "speech_start_grad": start_grad, "speech_end_grad": end_grad, "qwen_has_grad": qwen_has_grad, "encoder_has_grad": encoder_has_grad, "passed": True}
print("BACKWARD: PASS", phase2a["backward"], flush=True)
model.zero_grad(set_to_none=True)

## 11. Text-only generation parity: Hugging Face vs manual KV cache

In [ ]:
@torch.no_grad()
def manual_text_greedy(input_ids, max_new_tokens):
    attention_mask = torch.ones_like(input_ids, dtype=torch.bool)
    generated = []
    past = None
    current = input_ids
    for _ in range(max_new_tokens):
        position_ids = attention_mask.long().cumsum(-1) - 1
        outputs = model.llm(input_ids=current, attention_mask=attention_mask, position_ids=position_ids if past is None else position_ids[:, -1:], past_key_values=past, use_cache=True)
        past = outputs.past_key_values
        token = outputs.logits[:, -1].argmax(-1, keepdim=True)
        generated.append(token)
        current = token
        attention_mask = torch.cat([attention_mask, torch.ones_like(token, dtype=torch.bool)], dim=1)
    return torch.cat(generated, dim=1)

model.eval()
prompt_ids = tokenizer("The capital of France is", return_tensors="pt").input_ids.cuda()
N_TEXT_TOKENS = 8
hf_full = model.llm.generate(prompt_ids, max_new_tokens=N_TEXT_TOKENS, do_sample=False, use_cache=True, pad_token_id=tokenizer.eos_token_id)
hf_new = hf_full[:, prompt_ids.shape[1]:]
manual_new = manual_text_greedy(prompt_ids, N_TEXT_TOKENS)
assert torch.equal(hf_new, manual_new), (hf_new.tolist(), manual_new.tolist())
phase2a["text_generation_parity"] = {"tokens": hf_new[0].tolist(), "text": tokenizer.decode(hf_new[0]), "passed": True}
print("TEXT GENERATION + KV CACHE: PASS", phase2a["text_generation_parity"], flush=True)

## 12. Speech generation before training

In [ ]:
one = {k: v[0:1] for k,v in batch.items() if torch.is_tensor(v) and k in {"speech_states","speech_mask","prefix_ids","prefix_mask"}}
pretrain_ids = model.generate_cached(one, tokenizer.eos_token_id, max_new_tokens=16, use_kv_cache=True)[0]
assert len(pretrain_ids) <= 16
phase2a["speech_generation_pretrain"] = {"token_count": len(pretrain_ids), "ended_by": "eos" if len(pretrain_ids) < 16 else "max_new_tokens", "text": tokenizer.decode(pretrain_ids, skip_special_tokens=True), "passed": True}
print("SPEECH GENERATION PRETRAIN: PASS", phase2a["speech_generation_pretrain"], flush=True)

## 13. Cache integrity: live Stage 1 path vs saved cached states

In [ ]:
from aether_v3.data.stage2_cache import load_stage2_cache
record = load_stage2_cache(sorted((cache_root/"train").glob("shard-*.pt"))[0])[0]
codes = record["semantic_codes"].unsqueeze(0).cuda()
code_mask = torch.ones_like(codes, dtype=torch.bool)
with torch.no_grad():
    live = model.encoder(codes, code_mask)[0].cpu().to(torch.float16)
cached = record["speech_states"]
max_abs_diff = float((live - cached).abs().max())
allclose = bool(torch.allclose(live, cached, atol=1e-3, rtol=1e-3))
assert live.shape == cached.shape and allclose
phase2a["cache_integrity"] = {"shape": list(live.shape), "allclose": allclose, "max_abs_diff": max_abs_diff, "passed": True}
print("CACHE INTEGRITY: PASS", phase2a["cache_integrity"], flush=True)

## 14. Run one clean 5000-step Phase 2A R1 probe


In [ ]:
from aether_v3.training.train_stage2 import run_stage2_training

torch.cuda.reset_peak_memory_stats()
print("TRAIN: starting one clean 5000-step run; logs every 10 steps", flush=True)
run_stage2_training(cfg, run_dir, cache_root/"train", cache_root/"validation", model=model, tokenizer=tokenizer)
required = ["last.pt", "best_val_loss.pt", "best_wer.pt", "best_cer.pt", "periodic/step_005000.pt"]
for name in required:
    assert (run_dir/name).exists(), f"Missing artifact: {name}"
shutil.copy2(run_dir/"last.pt", run_dir/"phase2a_5000.pt")
print("TRAIN: completed; phase2a_5000.pt saved", flush=True)


## 15. Validate loss, gradients, memory, throughput, and post-training inference

In [ ]:
records = [json.loads(line) for line in (run_dir/"log.jsonl").read_text().splitlines()]
evals = [r for r in records if "val_loss" in r]
trains = [r for r in records if "train_loss" in r]
expected_eval_steps = [0, 500, 1000, 2000, 3000, 5000]
assert [r["step"] for r in evals] == expected_eval_steps
assert len(trains) == 500
assert len({r["step"] for r in trains}) == 500, "Duplicate training steps: run is not clean"
assert all(torch.isfinite(torch.tensor(r["train_loss"])) for r in trains)
assert all(r["grad_norm"] > 0 and r["steps_per_second"] > 0 for r in trains)
allocated = [r["gpu_allocated_gb"] for r in trains]
memory_growth_gb = allocated[-1] - allocated[0]
assert memory_growth_gb < 0.5, f"Possible VRAM leak: +{memory_growth_gb:.3f} GB"
l0 = evals[0]["val_loss"]
loss_reduction_2000 = (l0 - next(r["val_loss"] for r in evals if r["step"] == 2000)) / l0
loss_reduction_5000 = (l0 - evals[-1]["val_loss"]) / l0
phase2a["training"] = {
    "first_logged_loss": trains[0]["train_loss"], "last_logged_loss": trains[-1]["train_loss"],
    "eval_step0_loss": l0, "eval_step5000_loss": evals[-1]["val_loss"],
    "loss_reduction_2000": loss_reduction_2000, "loss_reduction_5000": loss_reduction_5000,
    "last_grad_norm": trains[-1]["grad_norm"], "bridge_output_scale_start": initial_scale,
    "bridge_output_scale_end": trains[-1]["bridge_output_scale"],
    "memory_growth_gb": memory_growth_gb, "last_steps_per_second": trains[-1]["steps_per_second"],
}
print("PHASE 2A TRAINING SUMMARY", json.dumps(phase2a["training"], indent=2), flush=True)


## 16. Full held-out evaluation on 256 fixed examples


In [ ]:
from aether_v3.eval.metrics import compute_cer, compute_wer

@torch.inference_mode()
def evaluate_full_split(split_name, cache_dir):
    loader = DataLoader(Stage2ShardDataset(cache_dir, shuffle=False), batch_size=1, num_workers=0, collate_fn=collate_stage2_batch)
    references, predictions, losses = [], [], []
    model.eval()
    started = time.time()
    for index, raw in enumerate(loader, start=1):
        current = {k: v.cuda() if torch.is_tensor(v) else v for k, v in raw.items()}
        output = model.forward_cached(current)
        losses.append(float(output.loss))
        generation = {k: current[k] for k in ("speech_states", "speech_mask", "prefix_ids", "prefix_mask")}
        target_length = int(current["target_mask"].sum())
        ids = model.generate_cached(generation, tokenizer.eos_token_id, max_new_tokens=min(256, max(32, target_length + 16)), use_kv_cache=True)[0]
        references.append(raw["references"][0][0])
        predictions.append(tokenizer.decode(ids, skip_special_tokens=True).strip())
        if index == 1 or index % 10 == 0:
            print(f"{split_name}: {index} examples | {index/max(time.time()-started, 1e-9):.2f} examples/sec", flush=True)
    result = {"examples": len(references), "loss": sum(losses)/len(losses), "wer": compute_wer(references, predictions), "cer": compute_cer(references, predictions)}
    print(split_name.upper(), json.dumps(result, indent=2), flush=True)
    return result

heldout_metrics = evaluate_full_split("heldout", cache_root/"validation")
phase2a["heldout_metrics"] = heldout_metrics
(run_dir/"phase2a_5000_heldout_metrics.json").write_text(json.dumps(heldout_metrics, indent=2))


## 17. Mandatory speech-dependence controls


In [ ]:
import itertools

control_records = list(itertools.islice(Stage2ShardDataset(cache_root/"validation", shuffle=False), 16))
conditions = {name: {"references": [], "predictions": []} for name in ("normal", "shuffled", "zero", "wrong", "truncated")}

@torch.inference_mode()
def decode_control(record, states, mask):
    raw = collate_stage2_batch([record])
    current = {k: v.cuda() if torch.is_tensor(v) else v for k,v in raw.items()}
    generation = {
        "speech_states": states,
        "speech_mask": mask,
        "prefix_ids": current["prefix_ids"],
        "prefix_mask": current["prefix_mask"],
    }
    ids = model.generate_cached(generation, tokenizer.eos_token_id, max_new_tokens=min(256, max(32, int(current["target_mask"].sum()) + 16)), use_kv_cache=True)[0]
    return raw["references"][0][0], tokenizer.decode(ids, skip_special_tokens=True).strip()

for index, record in enumerate(control_records):
    raw = collate_stage2_batch([record])
    states = raw["speech_states"].cuda()
    mask = raw["speech_mask"].cuda()
    length = int(mask.sum())
    permutation = torch.randperm(length, device=states.device)
    wrong_raw = collate_stage2_batch([control_records[(index + 1) % len(control_records)]])
    variants = {
        "normal": (states, mask),
        "shuffled": (states[:, permutation], mask),
        "zero": (torch.zeros_like(states), mask),
        "wrong": (wrong_raw["speech_states"].cuda(), wrong_raw["speech_mask"].cuda()),
        "truncated": (states[:, :max(1, length//2)], mask[:, :max(1, length//2)]),
    }
    for name,(variant_states,variant_mask) in variants.items():
        reference,prediction = decode_control(record,variant_states,variant_mask)
        conditions[name]["references"].append(reference)
        conditions[name]["predictions"].append(prediction)
    if index == 0 or (index + 1) % 4 == 0:
        print(f"controls: {index+1}/16", flush=True)

control_metrics = {}
for name,values in conditions.items():
    control_metrics[name] = {
        "wer": compute_wer(values["references"],values["predictions"]),
        "cer": compute_cer(values["references"],values["predictions"]),
        "examples": len(values["references"]),
    }
print(json.dumps(control_metrics,indent=2),flush=True)
phase2a["speech_controls"] = control_metrics
(run_dir/"phase2a_speech_controls.json").write_text(json.dumps(control_metrics,indent=2))


## 18. Final Phase 2A gate


In [ ]:
required_checks = ["cache_counts", "embedding_stats", "forward", "backward", "text_generation_parity", "speech_generation_pretrain", "cache_integrity", "training", "heldout_metrics", "speech_controls"]
assert all(name in phase2a for name in required_checks)
evals = [r for r in records if "val_loss" in r]
l0 = evals[0]["val_loss"]
loss_gate = phase2a["training"]["loss_reduction_5000"] >= 0.10
wer_values = [r["wer"] for r in evals]
wer_gate = wer_values[-1] <= 0.8 * wer_values[0]
controls = phase2a["speech_controls"]
speech_gate = all(controls[name]["wer"] > controls["normal"]["wer"] for name in ("shuffled", "zero", "wrong"))
phase2a["gate"] = {"loss_gate": loss_gate, "wer_gate": wer_gate, "speech_dependence_gate": speech_gate, "passed": bool((loss_gate or wer_gate) and speech_gate)}
phase2a["git_commit"] = GIT_COMMIT
phase2a["run_dir"] = str(run_dir)
phase2a["checkpoint"] = str(run_dir/"phase2a_5000.pt")
phase2a["status"] = "PASS" if phase2a["gate"]["passed"] else "FAIL"
(run_dir/"phase2a_report.json").write_text(json.dumps(phase2a,indent=2,ensure_ascii=False))
assert phase2a["gate"]["passed"], f"PHASE 2A FAILED: {phase2a['gate']}"
print("PHASE 2A: PASS",flush=True)
print("RUN:",run_dir,flush=True)
print("CHECKPOINT:",run_dir/"phase2a_5000.pt",flush=True)
